# Introducing Python for ABW — Part 2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/python/part-2.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/python/part-2.ipynb)

**2026/27 · Libraries, tables, plots and optimization · Tutorial 2 preparation**

Originally developed by the **ABW teaching team**. This edition preserves
the original Part 2 progression and the airline seat-allocation example.
It follows [ABW Python Part 1 — 2026/27](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/python/part-1.ipynb).

**Recall from Part 1:** basic types, functions, conditions, lists,
dictionaries, comprehensions and loops. Warm up, if useful, with
[CodingBat Warmup-1](https://codingbat.com/python/Warmup-1) and
[Warmup-2](https://codingbat.com/python/Warmup-2).

This part extends that foundation to imports, NumPy arrays, pandas tables,
notebook commands, plotting and a first optimization model. By the end,
explain which step transforms data, which displays data, and which chooses
a decision subject to constraints. The airline example is a **worked
example**, not an answer to submit as your own graded work.

Use **Open in Colab** or **Open in Binder** above and run from top to bottom.
Save a personal copy in Colab, or download your work before leaving Binder,
because Binder sessions are temporary. Predict first; use a fresh session and
**Run all** to check reproducibility. The
dependency checks below use installed packages and install only those
that are missing. They do not force upgrades or install a solver executable.

**Quick reference:** `name = value` binds a name; `==` compares values;
`items[i]` indexes a sequence; `mapping[key]` retrieves a dictionary value;
`def` defines a function; `return` supplies its result; indentation groups
statements. Do not confuse a label with a row or column position.

The earlier Part 2 notebook
is recorded for provenance. Use this edition for the revised lesson and
[Canvas](https://canvas.uva.nl/courses/62877) for current weekly instructions.


## Learn with AI without handing over the reasoning

AI can propose code; you need to judge what the code does and whether it answers
the question. Use the ABW verification loop: **attempt → ask for coaching → test
→ explain → record**. Predict the result before running a cell, then change one
input and explain what happens. A successful run is evidence about that run,
not a guarantee that the method is appropriate.

When course rules permit AI, use the **ABW Socratic Coach** in
[UvA AI Chat](https://aichat.uva.nl), following the persona in the course guide.
Choose **Concept coach**, **Code reviewer**, or **Model auditor**. For example:

> Mode: Code reviewer. Here is my own attempt and my prediction. Ask one focused
> question at a time and wait for my answer. Help me design a small test or a
> counterexample before suggesting a correction. Do not replace my attempt with
> a complete solution. Finish by asking me to explain what I checked.

These are self-study materials, not the graded submission. For graded work,
AI is forbidden unless the assignment explicitly permits it. Do not upload
personal or confidential data, unpublished assessment material, or answer keys.
Practise explaining code without AI or a computer: the ABW exam is invigilated
and completed on paper. Consult the current
[Canvas course](https://canvas.uva.nl/courses/62877) for assessment instructions.


## Import statement

For a right triangle with perpendicular sides `a` and `b`, the hypotenuse
has length $c = \sqrt{a^2 + b^2}$. The function below tries to use `sqrt`
without defining or importing it. **Predict the error first.**

This is an intentional error demonstration. The `try`/`except` wrapper
displays the expected error without stopping a top-to-bottom run; it is
not a way to ignore unexpected errors in your own analysis.


In [ ]:
def Pythagorean(a, b):
    c = sqrt(a**2 + b**2)
    return c

try:
    print(Pythagorean(3, 4))
except NameError as error:
    print(f'Expected NameError before importing sqrt: {error}')


`sqrt` is not defined in this session. Import the standard-library `math`
module and call `math.sqrt(...)`. The qualified name makes it explicit
where the function comes from. A module can define functions, constants
and other objects; no separate installation is needed for `math`.


In [ ]:
import math

def Pythagorean(a, b):
    c = math.sqrt(a**2 + b**2)
    return c

print(Pythagorean(3, 4))
assert math.isclose(Pythagorean(3, 4), 5.0)


Python code is organized in **modules**; **packages** organize related
modules. **Library** is a broader term for reusable functionality.
Installing a third-party package and importing a module are different steps.

We will use [NumPy](https://numpy.org/doc/stable/user/absolute_beginners.html)
for arrays and [pandas](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html)
for labelled tables. Common aliases are `np` and `pd`.

You can import a particular object with `from math import sqrt`, but
**avoid `from module import *`**: imported names may collide and their
origin becomes unclear. See [Python modules](https://docs.python.org/3/tutorial/modules.html).

The following setup checks for the packages used in the data/plotting
sections. If they are already installed, it makes no package changes.
A missing package is installed with `pip` into the current Python
environment; internet access is needed only for that installation.


In [ ]:
# Use installed packages, install only missing ones, without version pins.
from importlib.util import find_spec
import subprocess
import sys

required_packages = {
    'highspy': 'highspy',
    'matplotlib': 'matplotlib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyomo': 'pyomo',
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if find_spec(import_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])


In [ ]:
import numpy as np
import pandas as pd

# Import an individual object. A fixed seed makes this example reproducible.
from numpy.random import default_rng
rng = default_rng(2026)
x = rng.random(5)
print(x)


## NumPy

A NumPy `ndarray` is an array with a shape and a common data type (`dtype`).
`np.arange(15)` produces the integers from 0 through 14. Reshaping changes
how the entries are arranged, not how many entries there are.
Predict the shape and the last entry in the next example.


In [ ]:
# Example of an ndarray
a = np.arange(15)
print(a)

# We can reshape this ndarray to a 3*5 ndarray
a = a.reshape(3,5)
print(a)

In [ ]:
# It is possible to find the size and shape of the ndarray
print(a.shape)
print(a.size)

In [ ]:
assert a.shape == (3, 5)
assert a.size == 15
assert a[2, 4] == 14
print('Array shape, size and indexing checks passed.')


## Series and DataFrames

A pandas **Series** is one-dimensional data with an index. A **DataFrame**
is a two-dimensional table with row labels and column labels; different
columns may have different data types.

Labels carry meaning. `df['one']` selects a column, `df.loc[label]`
selects by row label, and `df.iloc[position]` selects by integer position.
Missing values are different from zeros. Watch how pandas aligns data
by label when creating a table from Series.


In [ ]:
s = pd.Series(rng.standard_normal(3), index=['a', 'b', 'c'])
print(s)


In [ ]:
# It is also possible to create series from a dictionary. Recall the dictionary from last week:
# Index is not passed, so the index will be ordered by dict's order
my_dict = {'name' : 'John', 'state':'Florida', 'age':26}
pd.Series(my_dict)

In [ ]:
# By passing the index, the order is changed
# The NaN (not a number) value means the data is missing
pd.Series(my_dict, index=["name","age","state","country"])

In [ ]:
# Create a nested dict and make it into a DataFrame
my_dict = {
          "one": pd.Series(["John", "26", "Florida"], index = ["name", "age", "state"]),
          "two": pd.Series(["Chris", "31", "Texas", "USA"], index = ["name", "age", "state", "country"])
}

df = pd.DataFrame(my_dict)
df

In [ ]:
# Pass index and columns to convert the part of the dictionary which you like to a DataFrame
df = pd.DataFrame(my_dict, index=["name", "age", "country"], columns = ["one","two","three"])
df

In [ ]:
# Select a specific column of the DataFrame
print(df["one"])

# Select a specific row of the DataFrame
print(df.loc["age"])

In [ ]:
# Find the content of a specific cell
print(df.loc["name", "three"])

# Fill a specific cell with new content (the cell doesn't have to exist already)
df.loc["age", "three"] = 25
df.loc["name", "four"] = "Leo"

df

### Check labels, positions and missing values

Before running the next cell, predict which label and position refer to
the same entry. What happens if rows are reordered? Why should a missing
age not automatically be replaced by zero?


In [ ]:
assert df.loc['age', 'three'] == df.iloc[1, 2] == 25
assert pd.isna(df.loc['name', 'three'])

reordered = df.iloc[::-1]
assert reordered.loc['age', 'three'] == 25
print('Original first row label:', df.index[0])
print('Reordered first row label:', reordered.index[0])
print('Label-based lookup still identifies the same age.')


In [ ]:
# Your turn: select one labelled entry and the same entry by position.
# Reorder columns; find the new position of that same labelled entry.
# Explain why a position from the old table need not identify the same data.


## Magics

**Magic commands** are supplied by IPython/Jupyter, not by the Python
language. `%` starts a line magic and `%%` starts a cell magic.
`%%time` reports the execution time of a cell; it must be its first line.
Timings vary by computer and session, so one timing is not a general
performance conclusion.
See [IPython magic commands](https://ipython.readthedocs.io/en/stable/interactive/magics.html).


In [ ]:
%%time
# A deliberately bounded computation for demonstrating cell timing.
x = 1
for i in range(5000):
    x = x * i + 1


In [ ]:
# Display Matplotlib figures inside this notebook.
%matplotlib inline


## Plotting figures

Matplotlib turns numbers into figures. Start a new figure for each example
so that plots do not accidentally accumulate when cells are rerun. Label
axes and check that the plotted quantities match those labels.
See the [official pyplot tutorial](https://matplotlib.org/stable/tutorials/pyplot.html).


In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.plot([2, 4, 6, 8])
plt.xlabel('Position (default x)')
plt.ylabel('Value (y)')
plt.show()


With four y-values and no explicit x-values, the plotted x-coordinates are
**0, 1, 2 and 3**, not 0 through 4. The next cell supplies x-values explicitly.
Predict the difference between the two plots.


In [ ]:
plt.figure()
plt.plot([1, 2, 3, 4], [2, 4, 6, 8])
plt.xlabel('x')
plt.ylabel('y')
plt.show()


A format string can set line colour and style. For example, `'r--'` means a
red dashed line. Prefer labels and line styles that remain interpretable
without colour. The [plot reference](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html)
describes the available options.


In [ ]:
plt.figure()
plt.plot([1, 2, 3, 4], [2, 4, 6, 8], 'r--')
plt.xlabel('x')
plt.ylabel('y')
plt.show()


In [ ]:
x1 = [1, 2, 3, 4]
y1 = [1, 2, 8, 16]
y2 = [1, 3, 9, 27]
y3 = [1, 4, 16, 64]

plt.figure()
plt.plot(x1, y1, 'r--', label='Series 1')
plt.plot(x1, y2, 'b:', label='Series 2')
plt.plot(x1, y3, 'g-', label='Series 3')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()


## Worked example: airline seat allocation


An airline decides how many regular-fare and discounted seats to sell on
a flight. **Fares are fixed inputs here; the decisions are seat quantities,
not ticket prices.** This is the original ABW example, with the same data.

| Symbol | Meaning | Value or bound |
| --- | --- | --- |
| $R$ | Regular-fare seats sold | $0 \le R \le 100$ |
| $D$ | Discounted seats sold | $0 \le D \le 150$ |
| Capacity | Total seats available | 166 seats |
| Regular fare | Revenue per regular seat | 617 currency units |
| Discount fare | Revenue per discounted seat | 238 currency units |

$$\max\;617R + 238D$$
$$R + D \le 166,\quad 0 \le R \le 100,\quad 0 \le D \le 150.$$

Both decisions are integers because seats are counted. The shaded polygon
below shows the **continuous relaxation**; feasible integer decisions are
the integer-coordinate points inside it. Put $D$ on the horizontal axis
and $R$ on the vertical axis, consistently.

Before solving: which fare is higher, and why can we not sell every seat
at that fare? Check one feasible and one infeasible decision by hand.


In [ ]:
# Horizontal axis: D (discounted seats). Vertical axis: R (regular seats).
D_axis = np.linspace(0, 170, 341)
D_feasible = np.linspace(0, 150, 301)
R_upper = np.minimum(100, 166 - D_feasible)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(D_axis, 166 - D_axis, label=r'$R + D \leq 166$')
ax.axhline(100, color='tab:orange', linestyle='--', label=r'$R \leq 100$')
ax.axvline(150, color='tab:green', linestyle=':', label=r'$D \leq 150$')
ax.fill_between(D_feasible, 0, R_upper, color='0.75', alpha=0.6,
                label='Feasible continuous relaxation')
ax.set(xlabel='D: discounted seats', ylabel='R: regular seats',
       xlim=(0, 170), ylim=(0, 175), title='Airline seat-allocation constraints')
ax.grid(alpha=0.25)
ax.legend(loc='upper right', fontsize=9)
plt.show()

# Check the plotted upper boundary against the model, not just its appearance.
assert np.all(D_feasible <= 150)
assert np.all((R_upper >= 0) & (R_upper <= 100))
assert np.all(D_feasible + R_upper <= 166 + 1e-9)
assert np.isclose(R_upper[0], 100)
assert np.isclose(R_upper[-1], 16)


### Expressing the same model in Pyomo

[Pyomo](https://www.pyomo.org/) expresses the variables, objective and
constraints. A **solver** searches for a decision that optimizes that
model. These are separate roles. This edition uses the open-source
**HiGHS** solver through its Python package, `highspy`; it does not require
CBC, a machine-specific executable path, or a commercial solver licence.

First check for missing packages, then import Pyomo and verify that the
solver is available. The setup does not upgrade working installations.
If importing or running an installed package fails, inspect the error
rather than automatically upgrading everything.

References: [Pyomo installation](https://pyomo.readthedocs.io/en/stable/getting_started/installation.html),
[Pyomo solver interfaces](https://pyomo.readthedocs.io/en/stable/reference/topical/appsi/appsi.html),
and the [ND Pyomo Cookbook](https://jckantor.github.io/ND-Pyomo-Cookbook/)
(additional examples; use this notebook's setup here).


Package presence is not sufficient: the solver must also be usable.
Initialize it every time this setup cell runs, including when the packages
were already installed. No system-wide `apt-get` or `conda` command is needed.


In [ ]:
import pyomo.environ as pyo

solver = pyo.SolverFactory('appsi_highs')
if not solver.available(exception_flag=False):
    raise RuntimeError('HiGHS is unavailable. Check the pyomo/highspy installation above.')
print('HiGHS is available.')


Read the function before running it. `model.x[1]` represents $R$ and
`model.x[2]` represents $D$, matching the original example. Identify the
units of each objective term and translate every constraint into words.

The solver's termination condition is checked **before** loading and
interpreting a result. A solver finding an optimum for the written model
does not establish that the model captures the intended business rules.


In [ ]:
def OptimizationModel(price_R, price_D, max_seats, max_R, max_D, solver=None):
    # Seat capacities must be non-negative whole numbers in this example.
    bounds = (max_seats, max_R, max_D)
    if any(bound < 0 or int(bound) != bound for bound in bounds):
        raise ValueError('Seat capacities must be non-negative integers.')

    if solver is None:
        solver = pyo.SolverFactory('appsi_highs')
    if not solver.available(exception_flag=False):
        raise RuntimeError('The requested solver is unavailable.')

    model = pyo.ConcreteModel()
    # x[1] = R, regular seats; x[2] = D, discounted seats.
    model.x = pyo.Var([1, 2], domain=pyo.NonNegativeIntegers)
    model.OBJ = pyo.Objective(
        expr=price_R * model.x[1] + price_D * model.x[2], sense=pyo.maximize)
    model.Constraint1 = pyo.Constraint(expr=model.x[1] + model.x[2] <= max_seats)
    model.Constraint2 = pyo.Constraint(expr=model.x[1] <= max_R)
    model.Constraint3 = pyo.Constraint(expr=model.x[2] <= max_D)

    result = solver.solve(model, load_solutions=False)
    if not pyo.check_optimal_termination(result):
        raise RuntimeError(f'No verified optimum: {result.solver.termination_condition}')
    model.solutions.load_from(result)

    revenue = pyo.value(model.OBJ)
    regular = pyo.value(model.x[1])
    discount = pyo.value(model.x[2])
    tolerance = 1e-6
    assert -tolerance <= regular <= max_R + tolerance
    assert -tolerance <= discount <= max_D + tolerance
    assert regular + discount <= max_seats + tolerance
    assert abs(regular - round(regular)) <= tolerance
    assert abs(discount - round(discount)) <= tolerance
    assert math.isclose(revenue, price_R * regular + price_D * discount,
                        rel_tol=1e-9, abs_tol=tolerance)
    return revenue, regular, discount


In [ ]:
revenue, R, D = OptimizationModel(617, 238, 166, 100, 150, solver=solver)
print(f'The optimal allocation has {R:.0f} regular seats and {D:.0f} discounted seats.')
print(f'Revenue: {revenue:,.0f} currency units.')


### Check independently: a small model permits complete enumeration

For this small instance, list every feasible pair of whole-number seat
quantities and calculate its revenue directly. This checks the solver
result using a different computation. It still cannot validate the assumed
fares or demand bounds against the real world.


In [ ]:
feasible_allocations = [
    (617 * regular + 238 * discount, regular, discount)
    for regular in range(101)
    for discount in range(151)
    if regular + discount <= 166
]
best_by_enumeration = max(feasible_allocations)
assert math.isclose(revenue, best_by_enumeration[0], rel_tol=1e-9)
assert (round(R), round(D)) == best_by_enumeration[1:]
print('The solver result agrees with complete enumeration of feasible integer decisions.')


## Your turn: change one assumption, then audit the result

1. Set the total capacity to zero. Predict the seat quantities and revenue.
2. Raise the regular-seat limit. Which constraint might become binding?
3. Swap the two fares. Predict how the allocation should change.
4. Explain why this is **optimization**, not a predictive model trained on
   past bookings. Where could a prediction of demand enter the model?
5. Identify a real-world assumption this small model does not represent.

> **ABW Socratic Coach — Model auditor:** Here are my formulation and my
> predicted response to a changed input. Ask me to state the decision
> variables, units, bounds and objective. Help me construct one feasible
> counterexample to a mistaken constraint. Ask one question at a time;
> do not replace my reasoning with a complete solution.

Use the coach only when permitted by the course/assignment rules. Then
close the notebook and explain the model and one result on paper.


In [ ]:
# Your turn: make a new call to OptimizationModel with one changed input.
# Write your prediction first, then check feasibility and interpret the result.
